In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib.colors import TwoSlopeNorm, Normalize
from matplotlib.patches import Rectangle

mpl.rcParams['font.family'] = 'Arial'

data_path = "PATH_TO_DATA"

seasons = ['MAM', 'JJA', 'SON', 'DJF']


def load_matrix(file_names, row_slices=((0, 30), (50, 60))):
    mean_values = []
    p_values = []
    climate_zones = None

    for file in file_names:
        df = pd.read_csv(os.path.join(data_path, file))

        if climate_zones is None:
            cz = [df.iloc[a:b, 1].values for (a, b) in row_slices]
            climate_zones = np.concatenate(cz)

        vals = [df.iloc[a:b, 2].values for (a, b) in row_slices]
        ps = [df.iloc[a:b, 3].values for (a, b) in row_slices]

        mean_values.append(np.concatenate(vals))
        p_values.append(np.concatenate(ps))

    mean_values = np.array(mean_values).T
    p_values = np.array(p_values).T
    return climate_zones.tolist(), mean_values, p_values


def process_zone_labels(climate_zones):
    out = []
    for zone in climate_zones:
        parts = str(zone).split('[')
        if len(parts) > 1:
            bracket_content = parts[1].split(')')[0]
            out.append(bracket_content.replace(',', ' -'))
        else:
            out.append(str(zone))
    return out


def insert_blank_rows(climate_zones, A, B, insert_positions=(10, 20, 30)):
    climate_zones = climate_zones.copy()
    A = A.copy()
    B = B.copy()
    for pos in sorted(insert_positions, reverse=True):
        climate_zones.insert(pos, '')
        A = np.insert(A, pos, np.nan, axis=0)
        B = np.insert(B, pos, np.nan, axis=0)
    return climate_zones, A, B


def reverse_all(climate_zones, A, B):
    return climate_zones[::-1], A[::-1, :], B[::-1, :]


def style_no_frame(ax):
    for sp in ax.spines.values():
        sp.set_visible(False)
    ax.tick_params(axis='both', which='both', length=0)


def draw_cell_grid(ax, ncols, nrows, cell_gap=0.5, lw=4):
    for i in range(ncols):
        for j in range(nrows):
            ax.add_patch(Rectangle((i - cell_gap, j - cell_gap),
                                   1 + 2 * cell_gap, 1 + 2 * cell_gap,
                                   linewidth=lw, edgecolor='white',
                                   facecolor='none', zorder=2))


def draw_sig_stars(ax, climate_zones, p_values, star_strong=19, star_weak=15):
    nrows, ncols = p_values.shape
    for i in range(ncols):
        for j in range(nrows):
            if climate_zones[i] == '':
                continue
            p = p_values[j, i]
            if np.isnan(p):
                continue
            if p < 0.05:
                ax.text(i, j, "*", ha='center', va='center',
                        fontsize=star_strong, fontweight='bold',
                        color='black', zorder=6)
            elif p < 0.1:
                ax.text(i, j, "*", ha='center', va='center',
                        fontsize=star_weak, fontweight='bold',
                        color='gray', zorder=6)


def get_group_blocks_by_nonblank(climate_zones, group_sizes=(10, 10, 10, 10)):
    nonblank_idx = [i for i, z in enumerate(climate_zones) if z != '']
    blocks = []
    cur = 0
    for gsz in group_sizes:
        seg = nonblank_idx[cur:cur + gsz]
        if len(seg) == 0:
            break
        blocks.append((min(seg), max(seg)))
        cur += gsz
    return blocks


def draw_group_bar_horizontal(ax, rect_y, rect_height, climate_zones,
                              group_labels=('Temperate', 'Tropical', 'Boreal', 'Arid'),
                              group_sizes=(10, 10, 10, 10)):
    blocks = get_group_blocks_by_nonblank(climate_zones, group_sizes=group_sizes)

    for k, (start, end) in enumerate(blocks):
        x0 = start - 0.5
        w = (end - start + 1)

        rect = Rectangle((x0, rect_y), w, rect_height,
                         edgecolor='white', facecolor='Gainsboro',
                         linewidth=2, linestyle='-',
                         zorder=3, clip_on=False)
        ax.add_patch(rect)

        label = group_labels[k] if k < len(group_labels) else f'Group {k+1}'
        ax.text(x0 + w / 2, rect_y + rect_height / 2, label,
                rotation=0, ha='center', va='center',
                fontsize=16, color='black', fontweight='bold',
                zorder=4, clip_on=False)


files_level = [
    "mean_area_proportion_Spring.csv",
    "mean_area_proportion_Summer.csv",
    "mean_area_proportion_Autumn.csv",
    "mean_area_proportion_Winter.csv",
]

files_change = [
    "rate_of_change_in_proportion_Spring.csv",
    "rate_of_change_in_proportion_Summer.csv",
    "rate_of_change_in_proportion_Autumn.csv",
    "rate_of_change_in_proportion_Winter.csv"
]

zones1, level, p_level = load_matrix(files_level)
zones2, change, p_change = load_matrix(files_change)

zones1 = process_zone_labels(zones1)
zones2 = process_zone_labels(zones2)

insert_positions = [10, 20, 30]
zones1, level, p_level = insert_blank_rows(zones1, level, p_level, insert_positions)
zones2, change, p_change = insert_blank_rows(zones2, change, p_change, insert_positions)

zones1, level, p_level = reverse_all(zones1, level, p_level)
zones2, change, p_change = reverse_all(zones2, change, p_change)

climate_zones = zones1

change_T = change.T
p_change_T = p_change.T

ncols = len(climate_zones)
nrows = len(seasons)

mean_all = np.nanmean(level, axis=1)

error_mode = "std"
if error_mode == "std":
    err_all = np.nanstd(level, axis=1)
else:
    err_all = np.nanmax(level, axis=1) - np.nanmin(level, axis=1)

mean_vmin, mean_vmax = 0.0, 0.6
s_min, s_max = 20, 200

err_vmin = 0.0
err_vmax = 0.07
norm_E = Normalize(vmin=err_vmin, vmax=err_vmax)
cmap_E = mpl.cm.inferno_r

vmax_R = 0.9
norm_R = TwoSlopeNorm(vmin=-vmax_R, vcenter=0.0, vmax=vmax_R)
cmap_R = mpl.cm.PuOr_r

cell_gap = 0.5
extent = [-cell_gap, ncols - 1 + cell_gap,
          nrows - 1 + cell_gap, -cell_gap]

fig, ax = plt.subplots(figsize=(14, 6))

imH = ax.imshow(change_T, cmap=cmap_R, norm=norm_R, aspect='equal',
                alpha=0.85, extent=extent)

bubble_y = -1.2
for i in range(ncols):
    if climate_zones[i] == '':
        continue

    mean_val = mean_all[i]
    err_val = err_all[i]

    if not (np.isfinite(mean_val) and np.isfinite(err_val)):
        continue

    m01 = (np.clip(mean_val, mean_vmin, mean_vmax) - mean_vmin) / (mean_vmax - mean_vmin + 1e-12)
    size = s_min + (s_max - s_min) * m01
    color = cmap_E(norm_E(np.clip(err_val, err_vmin, err_vmax)))

    bubble_x = i
    ax.scatter(bubble_x, bubble_y, s=size, c=[color],
               edgecolors='black', linewidths=0.8,
               zorder=10, clip_on=False)

ax.set_xticks(range(ncols))
ax.set_xticklabels(climate_zones, rotation=90, ha='center', fontsize=12)

ax.set_yticks(range(nrows))
ax.set_yticklabels(seasons, fontsize=14)

ax.tick_params(axis='x', pad=2)
ax.tick_params(axis='y', pad=6)

ax.set_xlim(-cell_gap, ncols - 1 + cell_gap)
ax.set_ylim(-1.8, nrows + 1.5)

style_no_frame(ax)

draw_cell_grid(ax, ncols=ncols, nrows=nrows, cell_gap=cell_gap, lw=4)
draw_sig_stars(ax, climate_zones, p_change_T)

bar_y_position = nrows - 1 + cell_gap + 0.3
draw_group_bar_horizontal(ax, rect_y=bar_y_position, rect_height=0.4,
                          climate_zones=climate_zones,
                          group_labels=('Temperate', 'Tropical', 'Boreal', 'Arid'),
                          group_sizes=(10, 10, 10, 10))

ax_pos = ax.get_position()

top_y = ax_pos.y1 + 0.06
cbar_height = 0.012
element_gap = 0.03

cbar1_left = ax_pos.x0
cbar1_width = 0.12
cax_bubble = fig.add_axes([cbar1_left, top_y, cbar1_width, cbar_height])
smE = mpl.cm.ScalarMappable(norm=norm_E, cmap=cmap_E)
cbE = fig.colorbar(smE, cax=cax_bubble, orientation='horizontal',
                   ticks=[err_vmin, err_vmax])
cbE.set_label('Seasonal STD', fontsize=14, labelpad=6)
cbE.ax.xaxis.set_label_position('top')
cbE.ax.xaxis.set_ticks_position('top')
cbE.ax.tick_params(axis='x', length=0, pad=1)
cbE.ax.set_xticklabels([f'{err_vmin:.2f}', f'{err_vmax:.2f}'], fontsize=12)

cbar2_left = cbar1_left + cbar1_width + element_gap
cbar2_width = 0.12
cax_heat = fig.add_axes([cbar2_left, top_y, cbar2_width, cbar_height])
cbH = fig.colorbar(imH, cax=cax_heat, orientation='horizontal',
                   ticks=[-vmax_R, 0, vmax_R], format='%0.1f')
cbH.set_label('Area Change Rate', fontsize=14, labelpad=6)
cbH.ax.xaxis.set_label_position('top')
cbH.ax.xaxis.set_ticks_position('top')
cbH.ax.tick_params(axis='x', length=0, pad=1)
cbH.ax.set_xticklabels([f'{-vmax_R:.1f}', '0', f'{vmax_R:.1f}'], fontsize=12)

legend_left = cbar2_left + cbar2_width + element_gap
legend_width = 0.20
legend_height = cbar_height

ax_legend = fig.add_axes([legend_left, top_y, legend_width, legend_height])
ax_legend.axis('off')

legend_vals = np.array([0.1, 0.3, 0.5])
legend_vals = legend_vals[(legend_vals >= mean_vmin) & (legend_vals <= mean_vmax)]
legend_sizes = s_min + (s_max - s_min) * ((legend_vals - mean_vmin) / (mean_vmax - mean_vmin + 1e-12))

handles, labels = [], []
for vv, ss in zip(legend_vals, legend_sizes):
    h = ax_legend.scatter([], [], s=ss, c=[cmap_E(norm_E(err_vmin))],
                          edgecolors='black', linewidths=0.8)
    handles.append(h)
    labels.append(f'{vv * 100:.0f}%')

ax_legend.legend(handles, labels, title='Mean Area Proportion',
                 frameon=False, fontsize=12, title_fontsize=14,
                 loc='center left', ncol=3,
                 handletextpad=0.3, columnspacing=0.8,
                 borderaxespad=0)

plt.subplots_adjust(left=0.075, right=0.95, top=0.88, bottom=0.22)

out = os.path.join("PATH_TO_OUTPUT", "OUTPUT_FIGURE.jpg")
plt.savefig(out, dpi=600, bbox_inches='tight', pad_inches=0.1)
plt.show()


In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib.colors import Normalize
from matplotlib.patches import Rectangle, Patch
import matplotlib.patheffects as pe
from matplotlib.lines import Line2D
from matplotlib.legend_handler import HandlerBase
from matplotlib.text import Text
from scipy.interpolate import PchipInterpolator

mpl.rcParams.update({
    "font.family": "Arial",
    "font.size": 12,
    "axes.titlesize": 12,
    "axes.labelsize": 12,
    "xtick.labelsize": 12,
    "ytick.labelsize": 12,
    "legend.fontsize": 12,
    "legend.title_fontsize": 12,
    "axes.linewidth": 0.8,
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "savefig.dpi": 600,
    "savefig.facecolor": "white",
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})

data_path_spearman = "PATH_TO_SPEI_STATS"
data_path_sm = "PATH_TO_SM_STATS"

seasons = ['MAM', 'JJA', 'SON', 'DJF']


def load_matrix(data_path, file_names, row_slices=((0, 21), (35, 42))):
    mean_values = []
    p_values = []
    climate_zones = None

    for file in file_names:
        df = pd.read_csv(os.path.join(data_path, file))

        if climate_zones is None:
            cz = [df.iloc[a:b, 1].values for (a, b) in row_slices]
            climate_zones = np.concatenate(cz)

        vals = [df.iloc[a:b, 2].values for (a, b) in row_slices]
        ps = [df.iloc[a:b, 3].values for (a, b) in row_slices]

        mean_values.append(np.concatenate(vals))
        p_values.append(np.concatenate(ps))

    mean_values = np.array(mean_values).T
    p_values = np.array(p_values).T
    return climate_zones.tolist(), mean_values, p_values


def process_zone_labels(climate_zones):
    out = []
    for zone in climate_zones:
        parts = str(zone).split('[')
        if len(parts) > 1:
            bracket_content = parts[1].split(')')[0]
            out.append(bracket_content.replace(',', ' -'))
        else:
            out.append(str(zone))
    return out


def insert_blank_rows(climate_zones, A, B, insert_positions=(7, 14, 21)):
    climate_zones = climate_zones.copy()
    A = A.copy()
    B = B.copy()
    for pos in sorted(insert_positions, reverse=True):
        climate_zones.insert(pos, '')
        A = np.insert(A, pos, np.nan, axis=0)
        B = np.insert(B, pos, np.nan, axis=0)
    return climate_zones, A, B


def style_no_frame(ax):
    for sp in ax.spines.values():
        sp.set_visible(False)
    ax.tick_params(axis='both', which='both', length=0)


def auto_absmax_scale(M, zones_cols=None, eps=1e-12):
    X = np.array(M, dtype=float, copy=True)

    if zones_cols is not None:
        blank_mask = np.array([z == '' for z in zones_cols])
        if blank_mask.any():
            X[:, blank_mask] = np.nan

    m = np.nanmax(np.abs(X))
    if not np.isfinite(m) or m <= 0:
        m = 1.0
    return float(m + eps)


def find_continuous_segments(mask):
    segments = []
    in_segment = False
    start = 0

    for i, valid in enumerate(mask):
        if valid and not in_segment:
            start = i
            in_segment = True
        elif not valid and in_segment:
            segments.append((start, i))
            in_segment = False

    if in_segment:
        segments.append((start, len(mask)))

    return segments


def draw_ridgeline(ax, M_rows_cols, zones_cols, scale=None, height=0.55,
                   line_color='#222222',
                   pos_color='#D95F02', neg_color="#483D8B",
                   alpha_fill=0.55, lw=1.15, zorder=2, label_prefix='',
                   use_pchip=True, pchip_points=300, linestyle='-'):
    nrows, ncols = M_rows_cols.shape
    x = np.arange(ncols)
    y_pos = np.full_like(M_rows_cols, np.nan, dtype=float)

    blank_mask = np.array([z == '' for z in zones_cols])

    if scale is None:
        scale = auto_absmax_scale(M_rows_cols, zones_cols=zones_cols)

    for j in range(nrows):
        v = M_rows_cols[j, :].astype(float).copy()
        v[blank_mask] = np.nan

        baseline = float(j)

        ax.plot([x[0] - 0.5, x[-1] + 0.5], [baseline, baseline],
                color='#ECECEC', lw=1.0, zorder=zorder - 1)

        valid_mask = np.isfinite(v)
        segments = find_continuous_segments(valid_mask)

        first_segment = True
        for seg_start, seg_end in segments:
            if seg_end - seg_start < 2:
                continue

            x_seg = x[seg_start:seg_end]
            v_seg = v[seg_start:seg_end]

            y_seg = baseline + height * (v_seg / scale)
            y_pos[j, seg_start:seg_end] = y_seg + 0.08

            label_line = f'{label_prefix}' if (j == 0 and label_prefix and first_segment) else None
            first_segment = False

            if use_pchip and len(x_seg) >= 3:
                interp = PchipInterpolator(x_seg, y_seg)
                x_smooth = np.linspace(x_seg[0], x_seg[-1], int(pchip_points))
                y_smooth = interp(x_smooth)

                ax.plot(x_smooth, y_smooth, color=line_color, lw=lw,
                        linestyle=linestyle, zorder=zorder + 1, label=label_line)

                y0 = np.full_like(y_smooth, baseline)
                y_pos_fill = np.where(y_smooth >= baseline, y_smooth, np.nan)
                y_neg_fill = np.where(y_smooth < baseline, y_smooth, np.nan)

                ax.fill_between(x_smooth, y0, y_pos_fill, where=np.isfinite(y_pos_fill),
                                color=pos_color, alpha=alpha_fill, zorder=zorder)
                ax.fill_between(x_smooth, y0, y_neg_fill, where=np.isfinite(y_neg_fill),
                                color=neg_color, alpha=alpha_fill, zorder=zorder)
            else:
                ax.plot(x_seg, y_seg, color=line_color, lw=lw,
                        linestyle=linestyle, zorder=zorder + 1, label=label_line)

                y0 = np.full_like(y_seg, baseline)
                y_pos_fill = np.where(v_seg >= 0, y_seg, np.nan)
                y_neg_fill = np.where(v_seg < 0, y_seg, np.nan)

                ax.fill_between(x_seg, y0, y_pos_fill, where=np.isfinite(y_pos_fill),
                                color=pos_color, alpha=alpha_fill, zorder=zorder)
                ax.fill_between(x_seg, y0, y_neg_fill, where=np.isfinite(y_neg_fill),
                                color=neg_color, alpha=alpha_fill, zorder=zorder)

    return y_pos, scale


def draw_sig_stars_dual(ax, zones_cols, p_rows_cols_1, p_rows_cols_2,
                        y_pos_rows_cols_1=None, y_pos_rows_cols_2=None,
                        star_strong_1=19, star_weak_1=15,
                        star_color_strong_1='black', star_color_weak_1='0.35',
                        star_symbol_1='*',
                        star_strong_2=14, star_weak_2=11,
                        star_color_strong_2='#2C7BB6', star_color_weak_2='#7FCDBB',
                        star_symbol_2='●',
                        offset=0.15, zorder_1=6, zorder_2=7):
    nrows, ncols = p_rows_cols_1.shape

    for i in range(ncols):
        if zones_cols[i] == '':
            continue

        for j in range(nrows):
            p1 = p_rows_cols_1[j, i]
            p2 = p_rows_cols_2[j, i]

            sig1 = np.isfinite(p1) and (p1 < 0.1)
            sig2 = np.isfinite(p2) and (p2 < 0.1)

            baseline = float(j)

            if sig1 and not sig2:
                yy = baseline if y_pos_rows_cols_1 is None else y_pos_rows_cols_1[j, i]
                if not np.isfinite(yy):
                    yy = baseline

                fs = star_strong_1 if p1 < 0.05 else star_weak_1
                col = star_color_strong_1 if p1 < 0.05 else star_color_weak_1
                t = ax.text(i, yy, star_symbol_1, ha='center', va='center',
                            fontsize=fs, fontweight='bold',
                            color=col, zorder=zorder_1)
                t.set_path_effects([pe.withStroke(linewidth=2.5, foreground="white")])

            elif sig2 and not sig1:
                yy = baseline if y_pos_rows_cols_2 is None else y_pos_rows_cols_2[j, i]
                if not np.isfinite(yy):
                    yy = baseline

                fs = star_strong_2 if p2 < 0.05 else star_weak_2
                col = star_color_strong_2 if p2 < 0.05 else star_color_weak_2
                t = ax.text(i, yy, star_symbol_2, ha='center', va='center',
                            fontsize=fs, fontweight='bold',
                            color=col, zorder=zorder_2)
                t.set_path_effects([pe.withStroke(linewidth=2.5, foreground="white")])

            elif sig1 and sig2:
                yy1 = baseline if y_pos_rows_cols_1 is None else y_pos_rows_cols_1[j, i]
                if not np.isfinite(yy1):
                    yy1 = baseline
                yy1 = max(yy1, baseline) + offset * 0.3

                fs1 = star_strong_1 if p1 < 0.05 else star_weak_1
                col1 = star_color_strong_1 if p1 < 0.05 else star_color_weak_1
                t1 = ax.text(i, yy1, star_symbol_1, ha='center', va='center',
                             fontsize=fs1, fontweight='bold',
                             color=col1, zorder=zorder_1)
                t1.set_path_effects([pe.withStroke(linewidth=2.5, foreground="white")])

                yy2 = baseline if y_pos_rows_cols_2 is None else y_pos_rows_cols_2[j, i]
                if not np.isfinite(yy2):
                    yy2 = baseline
                yy2 = min(yy2, baseline) - offset * 0.3

                fs2 = star_strong_2 if p2 < 0.05 else star_weak_2
                col2 = star_color_strong_2 if p2 < 0.05 else star_color_weak_2
                t2 = ax.text(i, yy2, star_symbol_2, ha='center', va='center',
                             fontsize=fs2, fontweight='bold',
                             color=col2, zorder=zorder_2)
                t2.set_path_effects([pe.withStroke(linewidth=2.5, foreground="white")])


def draw_group_bar_horizontal_auto(ax, zones_cols, rect_y, rect_height,
                                   group_labels=('Temperate', 'Tropical', 'Boreal', 'Arid'),
                                   fontsize=12):
    n = len(zones_cols)
    blanks = [i for i, z in enumerate(zones_cols) if z == '']

    starts = [0] + [b + 1 for b in blanks]
    ends = blanks + [n]

    rect_params = []
    for s, e in zip(starts, ends):
        if e <= s:
            continue
        x0 = s - 0.5
        w = e - s
        rect_params.append((x0, rect_y, w, rect_height))

    for x, y, w, h in rect_params:
        ax.add_patch(Rectangle((x, y), w, h,
                               edgecolor='white', facecolor='0.94',
                               linewidth=2, linestyle='-',
                               zorder=3, clip_on=False))

    k = min(len(rect_params), len(group_labels))
    for i in range(k):
        x, y, w, h = rect_params[i]
        ax.text(x + w / 2, y + h / 2, group_labels[i],
                rotation=0, ha='center', va='center',
                fontsize=fontsize, color='black', fontweight='bold',
                zorder=4, clip_on=False)


files_level_spearman = [
    "mean_area_proportion_Ts_Spring.csv",
    "mean_area_proportion_Ts_Summer.csv",
    "mean_area_proportion_Ts_Autumn.csv",
    "mean_area_proportion_Ts_Winter.csv",
]

files_change_spearman = [
    "rate_of_change_in_proportion_Ts_Spring.csv",
    "rate_of_change_in_proportion_Ts_Summer.csv",
    "rate_of_change_in_proportion_Ts_Autumn.csv",
    "rate_of_change_in_proportion_Ts_Winter.csv"
]

files_change_sm = [
    "rate_of_change_in_proportion_Ts_Spring.csv",
    "rate_of_change_in_proportion_Ts_Summer.csv",
    "rate_of_change_in_proportion_Ts_Autumn.csv",
    "rate_of_change_in_proportion_Ts_Winter.csv"
]

files_level_sm = [
    "mean_area_proportion_Ts_Spring.csv",
    "mean_area_proportion_Ts_Summer.csv",
    "mean_area_proportion_Ts_Autumn.csv",
    "mean_area_proportion_Ts_Winter.csv",
]

row_slices = ((0, 21), (35, 42))

zones1, level_spearman, p_level_spearman = load_matrix(data_path_spearman, files_level_spearman, row_slices=row_slices)
zones2, change_spearman, p_change_spearman = load_matrix(data_path_spearman, files_change_spearman, row_slices=row_slices)

zones3, change_sm, p_change_sm = load_matrix(data_path_sm, files_change_sm, row_slices=row_slices)
zones4, level_sm, p_level_sm = load_matrix(data_path_sm, files_level_sm, row_slices=row_slices)

zones1 = process_zone_labels(zones1)
zones2 = process_zone_labels(zones2)
zones3 = process_zone_labels(zones3)
zones4 = process_zone_labels(zones4)

insert_positions = [7, 14, 21]
zones1, level_spearman, p_level_spearman = insert_blank_rows(zones1, level_spearman, p_level_spearman, insert_positions)
zones2, change_spearman, p_change_spearman = insert_blank_rows(zones2, change_spearman, p_change_spearman, insert_positions)
zones3, change_sm, p_change_sm = insert_blank_rows(zones3, change_sm, p_change_sm, insert_positions)
zones4, level_sm, p_level_sm = insert_blank_rows(zones4, level_sm, p_level_sm, insert_positions)

climate_zones = zones1

change_spearman_T = change_spearman.T
p_change_spearman_T = p_change_spearman.T

change_sm_T = change_sm.T
p_change_sm_T = p_change_sm.T

zones_cols = climate_zones[::-1]
change_spearman_plot = change_spearman_T[:, ::-1]
p_spearman_plot = p_change_spearman_T[:, ::-1]

change_sm_plot = change_sm_T[:, ::-1]
p_sm_plot = p_change_sm_T[:, ::-1]

ncols = len(zones_cols)
nrows = len(seasons)

mean_spei = np.nanmean(level_spearman, axis=1)
err_spei = np.nanstd(level_spearman, axis=1)
mean_spei_plot = mean_spei[::-1]
err_spei_plot = err_spei[::-1]

mean_smrz = np.nanmean(level_sm, axis=1)
err_smrz = np.nanstd(level_sm, axis=1)
mean_smrz_plot = mean_smrz[::-1]
err_smrz_plot = err_smrz[::-1]

mean_vmin, mean_vmax = 0.0, 0.6
s_min, s_max = 20, 200

err_vmin, err_vmax = 0.0, 0.07
norm_E = Normalize(vmin=err_vmin, vmax=err_vmax)
cmap_E = mpl.cm.inferno_r

vmax_spearman = auto_absmax_scale(change_spearman_plot, zones_cols=zones_cols)
vmax_sm = auto_absmax_scale(change_sm_plot, zones_cols=zones_cols)
vmax_R = max(vmax_spearman, vmax_sm)

fig = plt.figure(figsize=(14, 3))
gs = fig.add_gridspec(nrows=4, ncols=1,
                      height_ratios=[0.4, 0.5, 5.5, 0.6],
                      hspace=0.3)

ax_group = fig.add_subplot(gs[0, 0])
ax_bub_smrz = fig.add_subplot(gs[1, 0], sharex=ax_group)
ax_rid = fig.add_subplot(gs[2, 0], sharex=ax_group)
ax_bub_spei = fig.add_subplot(gs[3, 0], sharex=ax_group)

style_no_frame(ax_group)
ax_group.set_ylim(0, 1)
ax_group.set_yticks([])
ax_group.tick_params(axis='x', labelbottom=False)
ax_group.set_xlim(-0.5, ncols - 0.5)

draw_group_bar_horizontal_auto(
    ax_group, zones_cols, rect_y=0.2, rect_height=0.6,
    group_labels=('Temperate', 'Tropical', 'Boreal', 'Arid'),
    fontsize=16
)

style_no_frame(ax_bub_smrz)
ax_bub_smrz.set_ylim(-0.5, 0.5)
ax_bub_smrz.set_yticks([])
ax_bub_smrz.tick_params(axis='x', labelbottom=False)

ax_bub_smrz.text(-0.5, 0, 'SM_rz', ha='right', va='center', fontsize=14)

for i in range(ncols):
    if zones_cols[i] == '':
        continue
    mv = mean_smrz_plot[i]
    ev = err_smrz_plot[i]
    if not (np.isfinite(mv) and np.isfinite(ev)):
        continue

    m01 = (np.clip(mv, mean_vmin, mean_vmax) - mean_vmin) / (mean_vmax - mean_vmin + 1e-12)
    size = s_min + (s_max - s_min) * m01
    color = cmap_E(norm_E(np.clip(ev, err_vmin, err_vmax)))

    ax_bub_smrz.scatter(i, 0.0, s=size, c=[color],
                        edgecolors='black', linewidths=0.8, zorder=5)

RIDGE_HEIGHT = 0.75

y_star_pos_spearman, _ = draw_ridgeline(
    ax_rid, change_spearman_plot, zones_cols,
    scale=vmax_R,
    height=RIDGE_HEIGHT,
    line_color="#964800",
    pos_color='#D95F02',
    neg_color='#483D8B',
    alpha_fill=0.55,
    lw=1.15,
    zorder=2,
    label_prefix='Spearman',
    use_pchip=True,
    pchip_points=350,
    linestyle='-'
)

y_star_pos_sm, _ = draw_ridgeline(
    ax_rid, change_sm_plot, zones_cols,
    scale=vmax_R,
    height=RIDGE_HEIGHT,
    line_color="#420097",
    pos_color="#D95F02",
    neg_color='#483D8B',
    alpha_fill=0.45,
    lw=1.2,
    zorder=4,
    label_prefix='SM',
    use_pchip=True,
    pchip_points=350,
    linestyle='--'
)

draw_sig_stars_dual(
    ax_rid, zones_cols,
    p_spearman_plot, p_sm_plot,
    y_star_pos_spearman, y_star_pos_sm,
    star_strong_1=19, star_weak_1=15,
    star_color_strong_1='black', star_color_weak_1='0.35',
    star_symbol_1='*',
    star_strong_2=14, star_weak_2=11,
    star_color_strong_2='#2C7BB6', star_color_weak_2='#7FCDBB',
    star_symbol_2='●',
    offset=0.15,
    zorder_1=6, zorder_2=7
)

style_no_frame(ax_rid)
ax_rid.set_yticks(range(nrows))
ax_rid.set_yticklabels(seasons, fontsize=14)
ax_rid.tick_params(axis='x', labelbottom=False)
ax_rid.set_xlim(-0.5, ncols - 0.5)
ax_rid.set_ylim(-0.5, nrows - 0.1)

style_no_frame(ax_bub_spei)
ax_bub_spei.set_ylim(-0.6, 0.6)
ax_bub_spei.set_yticks([])

ax_bub_spei.set_xticks(range(ncols))
ax_bub_spei.set_xticklabels(zones_cols, rotation=90, ha='center', fontsize=12)

ax_bub_spei.text(-0.5, 0, 'SPEI', ha='right', va='center', fontsize=14)

for i in range(ncols):
    if zones_cols[i] == '':
        continue
    mv = mean_spei_plot[i]
    ev = err_spei_plot[i]
    if not (np.isfinite(mv) and np.isfinite(ev)):
        continue

    m01 = (np.clip(mv, mean_vmin, mean_vmax) - mean_vmin) / (mean_vmax - mean_vmin + 1e-12)
    size = s_min + (s_max - s_min) * m01
    color = cmap_E(norm_E(np.clip(ev, err_vmin, err_vmax)))

    ax_bub_spei.scatter(i, 0.0, s=size, c=[color],
                        edgecolors='black', linewidths=0.8, zorder=5)


class TextStarHandler(HandlerBase):
    def __init__(self, color='black', size_multiplier=1.0):
        self.color = color
        self.size_multiplier = size_multiplier
        super().__init__()

    def create_artists(self, legend, orig_handle, xdescent, ydescent,
                       width, height, fontsize, trans):
        x = width / 2
        y = height / 2
        actual_fontsize = fontsize * self.size_multiplier
        text = Text(x, y, '*', fontsize=actual_fontsize,
                    ha='center', va='center', fontweight='bold',
                    color=self.color, transform=trans)
        return [text]


class TextCircleHandler(HandlerBase):
    def __init__(self, color='#2C7BB6', size_multiplier=1.0):
        self.color = color
        self.size_multiplier = size_multiplier
        super().__init__()

    def create_artists(self, legend, orig_handle, xdescent, ydescent,
                       width, height, fontsize, trans):
        x = width / 2
        y = height / 2
        actual_fontsize = fontsize * self.size_multiplier
        text = Text(x, y, '●', fontsize=actual_fontsize,
                    ha='center', va='center', fontweight='bold',
                    color=self.color, transform=trans)
        return [text]


dummy_star_strong = Rectangle((0, 0), 1, 1, facecolor='none', edgecolor='none')
dummy_star_weak = Rectangle((0, 0), 1, 1, facecolor='none', edgecolor='none')
dummy_circle_strong = Rectangle((0, 0), 1, 1, facecolor='none', edgecolor='none')
dummy_circle_weak = Rectangle((0, 0), 1, 1, facecolor='none', edgecolor='none')

legend_elements = [
    Line2D([0], [0], color='#964800', lw=1.15, linestyle='-'),
    Line2D([0], [0], color='#420097', lw=1.2, linestyle='--'),
    Patch(facecolor='#D95F02', alpha=0.55),
    Patch(facecolor='#483D8B', alpha=0.55),
    dummy_star_strong,
    dummy_star_weak,
    dummy_circle_strong,
    dummy_circle_weak,
]

legend_labels = [
    'SPEI-SIF',
    'SMrz-SIF',
    'Positive',
    'Negative',
    'SPEI-SIF p<0.05',
    'SPEI-SIF p<0.1',
    'SMrz-SIF p<0.05',
    'SMrz-SIF p<0.1'
]

legend_fontsize = 12

fig.legend(
    handles=legend_elements,
    labels=legend_labels,
    handler_map={
        dummy_star_strong: TextStarHandler(color='black', size_multiplier=19 / legend_fontsize),
        dummy_star_weak: TextStarHandler(color='0.35', size_multiplier=15 / legend_fontsize),
        dummy_circle_strong: TextCircleHandler(color='#2C7BB6', size_multiplier=14 / legend_fontsize),
        dummy_circle_weak: TextCircleHandler(color='#7FCDBB', size_multiplier=11 / legend_fontsize)
    },
    loc='upper center',
    bbox_to_anchor=(0.75, 1.25),
    ncol=4,
    frameon=False,
    fontsize=legend_fontsize,
    columnspacing=1.0,
    handlelength=1.5,
    handletextpad=0.5
)

plt.subplots_adjust(left=0.08, right=0.97, top=0.96, bottom=0.15)

out = os.path.join("PATH_TO_OUTPUT", "OUTPUT_FIGURE.png")
plt.savefig(out, dpi=600, bbox_inches='tight', pad_inches=0.1)
plt.show()
